# method

> The ways you learn: six text ways (code, maths, analogies, worked examples, diagrams and plain words), and demos, ordering, playgrounds, boards and timelines. This module finds which ways work for you, and when to try a new one.

Each question, hint and lesson uses one way to teach: its `method`. The tutor picks the method before it writes the prompt. So the tag is
known, not guessed. When a question closes, its outcome counts for its method: solid 1, shaky 0.5, a miss 0.

The counts give each method a Beta posterior, for each subject and over all subjects. `choose` uses them in two ways:

- Drill: a weak concept gets your best method.
- Explore: a concept that is not weak sometimes gets another method. You set the chance (`explore` in the profile). First `choose` tries
  a method that got better recently. Else it uses a random draw (a Thompson sample).

The counts are in one `stats` document on the shelf. This document is only a cache: `method_stats` builds it again from the attempts.

In [ ]:
#| default_exp method

In [ ]:
#| hide
import os, tempfile
from fastcore.test import *
from abhyasa.store import temp_learning
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the saved prompts go to a temporary folder
from abhyasa.prompts import PROMPTS

Import the modules that this notebook uses.

In [ ]:
#| export
import json, random, threading, time
from fastcore.utils import *
from abhyasa.store import Learning, _src
from abhyasa.prompts import register, prompt

## The methods

`METHODS` maps each method to the line that the prompt gets. `EXAMPLES_TO` maps the profile's "examples that help" to methods, to set the prior.

In [ ]:
#| export
METHODS = dict(code='Use a short Python code example that shows the idea; keep it under 12 lines.',
               maths='Use equations and formal notation; derive the step, one line at a time.',
               analogy="Use an analogy from the learner's own world (see 'About the learner'), and say where it stops working.",
               example='Use one concrete worked example, step by step, with real numbers or real words.',
               diagram='Describe a picture in words: what to draw, where each part goes, and what moves.',
               words='Use plain words only: no code and no equations.',
               demo='Show a small figure that the learner can change. Ask for a prediction first, then show the result.',
               order='Give the steps of the idea in a mixed order. Ask the learner to put them in the right order.',
               playground='Give a small space where the learner changes one value and sees what happens.',
               board='Give a board with the parts of the idea. Ask the learner to put each part where it belongs.',
               timeline='Give a line of events for the idea. Ask the learner to put the events in time order.')
LABELS = dict(code='Code', maths='Maths', analogy='Analogies', example='Worked examples', diagram='Pictures', words='Plain words',
          demo='Demos', order='Put in order', playground='Playgrounds', board='Boards', timeline='Timelines')
EXAMPLES_TO = {'code': 'code', 'maths': 'maths', 'analogies': 'analogy', 'stories': 'analogy', 'diagrams': 'diagram', 'real cases': 'example'}
TEXT_WAYS = ('code', 'maths', 'analogy', 'example', 'diagram', 'words')   # the ways that every session can show
WALK_WAYS = TEXT_WAYS + ('demo', 'order')                                    # the ways that a walk can show now
CREDIT = dict(solid=1., fragile=.5, gap=0., misconception=0.)
FEEDBACK = dict(unclear=('gap', .25), right=('solid', .1))   # a verdict on a demo or a lesson -> the outcome and the weight that it counts for its way
EXPLORE = dict(never=0., sometimes=.15, often=.3)
_SRC, _LOCK = _src('stats', 'me'), threading.Lock()

The text of each method is a prompt that you can change in Settings (see `prompts`). `way` gives the text in use: your saved text, else the line in `METHODS`.

In [ ]:
#| export
for _m, _t in METHODS.items(): register(f'way.{_m}', f'Way: {LABELS[_m]}', 'How the tutor uses this way in a question, a nudge, a hint and a lesson.', _t)

def way(m:str) -> str:
    "The text of method `m` now: your saved text, else the line in `METHODS`; '' for an unknown method."
    return prompt(f'way.{m}') if m in METHODS else ''

With no saved text, `way` gives the line in `METHODS`.

In [ ]:
test_eq([way(m) for m in METHODS], list(METHODS.values()))
test_eq(way('nope'), '')

## The stats document

`stats` reads the stats document. It gives empty counts when there is no document. `set_stats` writes the document. A lock makes sure that two closes at the same time do not lose a count.

In [ ]:
#| export
EMPTY = dict(methods={}, recent={}, calib={})

@patch
def stats(self:Learning) -> AttrDict:
    "The cached stats: method counts for each subject (`*` is all), recent credits, and calibration."
    r = first(self.v.t.docs(where='source=?', where_args=[_SRC]))
    return AttrDict({**EMPTY, **(json.loads(r['meta'] or '{}') if r else {})})

@patch
def set_stats(self:Learning, **kv):
    "Change fields of the stats document."
    m = {**{k: v for k, v in self.stats().items() if k in EMPTY}, **kv}
    r = self.v.add('How you learn: method counts.', title='How you learn', source=_SRC, kind='stats', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)

## Credit

`credit` adds one outcome to the counts and returns the new stats. It keeps the last 8 credits of each method, to find a method that gets better.

In [ ]:
#| export
def credit(st, method:str, subject:str, kind:str, w:float=1.) -> dict:
    "The stats `st` with one outcome `kind` added for `method` with the weight `w`, in `subject` and in `*`. Only a full outcome (`w=1`) goes in the recent credits."
    s, ms = CREDIT.get(kind), {k: {m: list(ab) for m, ab in v.items()} for k, v in (st.get('methods') or {}).items()}
    if s is None or method not in METHODS: return dict(st)
    for k in {subject or '*', '*'}:
        a, b = ms.setdefault(k, {}).get(method, [0., 0.])
        ms[k][method] = [a + w * s, b + w * (1 - s)]
    rc = {**(st.get('recent') or {})}
    if w == 1: rc[method] = [*rc.get(method, []), s][-8:]
    return {**st, 'methods': ms, 'recent': rc}

@patch
def credit_method(self:Learning, method:str, subject:str, kind:str, w:float=1.):
    "Count outcome `kind` for `method` with the weight `w` in the stats document."
    with _LOCK: self.set_stats(**{k: v for k, v in credit(self.stats(), method, subject, kind, w).items() if k in EMPTY})

Solid adds 1 to `a`. A miss adds 1 to `b`. Shaky adds half to each. An unknown outcome or method changes nothing.

In [ ]:
st = credit(credit(credit(EMPTY, 'code', 'ml', 'solid'), 'code', 'ml', 'fragile'), 'maths', 'ml', 'gap')
test_eq(st['methods']['ml'], {'code': [1.5, .5], 'maths': [0., 1.]})
test_eq(st['methods']['*'], st['methods']['ml'])
test_eq(st['recent'], {'code': [1., .5], 'maths': [0.]})
test_eq(credit(st, 'code', 'ml', 'closed'), st)

### Feedback counts a little

The row under a demo or a lesson also counts for the way of that demo or lesson, with a small weight from `FEEDBACK`. "Not clear" adds 0.25 of a miss, so four of them count as one miss, and one click does not decide the way. "Just right" adds 0.1 of a pass. "Too simple" adds nothing: the way worked, but the step was too small. A part of an outcome does not go in the recent credits, so it does not make a way look better lately.

In [ ]:
st = credit(EMPTY, 'analogy', 'ml', *FEEDBACK['unclear'])
test_eq(st['methods']['ml']['analogy'], [0., .25])
test_eq(st['recent'], {})
test_eq(credit(st, 'analogy', 'ml', *FEEDBACK['right'])['methods']['ml']['analogy'], [.1, .25])

## The posterior

`posterior` gives `(a, b)` for a method: Beta(1, 1), plus 2 on `a` for a method in the profile's examples. For a subject, it adds the subject counts and the `*` counts. It scales the `*` counts down to at most 4 pseudo-counts. So a new subject starts from what works for you in other subjects.

In [ ]:
#| export
def posterior(st, m:str, subject:str='', prior=()) -> tuple:
    "The Beta `(a, b)` of method `m` in `subject`."
    ms = st.get('methods') or {}
    a, b = 1. + 2 * (m in prior), 1.
    g = (ms.get('*') or {}).get(m, [0., 0.])
    if subject and (s := (ms.get(subject) or {}).get(m)):
        k = min(1., 4 / max(sum(g), 1e-9))
        return a + s[0] + k * g[0], b + s[1] + k * g[1]
    return a + g[0], b + g[1]

def mean(ab) -> float: return ab[0] / sum(ab)

def improving(st, m:str) -> tuple|None:
    "`(right, n)` over the last 5 credits of `m`, when they are at least 0.2 above its mean over all; else None."
    r, ab = (st.get('recent') or {}).get(m, []), (st.get('methods') or {}).get('*', {}).get(m)
    if len(r) < 4 or not ab or sum(ab) < 4: return None
    last = r[-5:]
    return (round(sum(last)), len(last)) if sum(last) / len(last) >= ab[0] / sum(ab) + .2 else None

With no counts, the prior decides. `improving` finds a method that got better recently.

In [ ]:
test_eq(posterior(EMPTY, 'code'), (1., 1.))
test_eq(posterior(EMPTY, 'code', prior=['code']), (3., 1.))
gm = dict(methods={'*': {'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
test_eq(improving(gm, 'maths'), (4, 5))
test_is(improving(dict(gm, recent={'maths': [1., 0., 0., 0.]}), 'maths'), None)

## Choose a method

`choose` returns `(method, note)`. The note goes in the line that says why you get this question.

In [ ]:
#| export
def _count(st, m, subject=''):
    "The raw `(right, n)` of method `m`, in `subject` when it has counts there, else over all."
    ms = st.get('methods') or {}
    a, b = (ms.get(subject) or {}).get(m) or (ms.get('*') or {}).get(m) or [0., 0.]
    return round(a), round(a + b)

def choose(st, subject:str='', weak:bool=False, rng=None, explore:str='sometimes', prior=(), allowed=None) -> tuple:
    "The method for the next question, and a note on why, from the `allowed` ways (default `TEXT_WAYS`). A weak concept gets the best method; else, sometimes, another one."
    rng = rng or random
    cands = [m for m in allowed or TEXT_WAYS if m in METHODS] or list(TEXT_WAYS)
    ps = {m: posterior(st, m, subject, prior) for m in cands}
    best = max(ps, key=lambda m: (mean(ps[m]), m in prior))
    if weak:
        k, n = _count(st, best, subject)
        return best, f"drill with {LABELS[best].lower()}: {k} of {n}, your best way" if n else (f"drill with {LABELS[best].lower()}: a way that you said helps" if best in prior else '')
    if (up := first((m, improving(st, m)) for m in cands if m != best and improving(st, m))):
        m, (k, n) = up
        if rng.random() < max(EXPLORE.get(explore, .15) * 2, .3) * bool(EXPLORE.get(explore, .15)):
            return m, f"trying {LABELS[m].lower()}: you got better with it ({k} of {n} lately)"
    if (others := [m for m in cands if m != best]) and rng.random() < EXPLORE.get(explore, .15):
        m = max(others, key=lambda m: rng.betavariate(*ps[m]))
        return m, f"trying {LABELS[m].lower()}: a new way to learn this"
    return best, ''

A weak concept gets the best method. With `explore='never'`, `choose` always gives the best method. With `sometimes`, it tries a method that got better at times. It still gives the best method most of the time.

In [ ]:
cs = dict(methods={'*': {'code': [7., 2.], 'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
m, note = choose(cs, 'ml', weak=True)
test_eq(m, 'code'); assert note.startswith('drill with code: 7 of 9')
test_eq({choose(cs, 'ml', rng=random.Random(i), explore='never')[0] for i in range(30)}, {'code'})
picks = [choose(cs, 'ml', rng=random.Random(i)) for i in range(60)]
assert any(n.startswith('trying maths: you got better') for _, n in picks)
test_eq(sum(m == 'code' for m, _ in picks) > 30, True)   # with `sometimes`, the best method still comes most of the time

## The ways that fit

A session cannot show every way. `choose` takes `allowed`: the ways it may pick from. The default is `TEXT_WAYS`, the ways that every session can show. A list with no known way gives `TEXT_WAYS` too.

In [ ]:
rng = random.Random(0)
assert {'demo', 'order', 'playground', 'board', 'timeline'} <= set(METHODS) and 'way.timeline' in PROMPTS
assert all(choose(EMPTY, rng=rng)[0] in TEXT_WAYS for _ in range(200))          # a session never gets a way it cannot show
assert all(choose(EMPTY, rng=rng, allowed=['demo', 'maths'])[0] in ('demo', 'maths') for _ in range(200))
assert choose(EMPTY, rng=rng, allowed=['board'])[0] in METHODS                   # an allowed list keeps its own ways
assert choose(EMPTY, rng=rng, allowed=['nope'])[0] in TEXT_WAYS                  # no known way: back to the text ways

## The way for a check

`check_for` picks the way for one check of one idea. It looks only at the ways of the idea that a walk can show. It does not repeat the way of the last check when it has another. The kind of the check follows the way, and the depth follows the kind.

A hands-on way of the idea (a key of `CHECK_KIND`: demo, order or code) comes before its text ways. The model lists a hands-on way only for an idea that fits it: a process, a sequence, or code that runs with the standard library. Without this rule, the learner stats and the profile pick a text way almost every time, because the profile names only text ways. Then nearly every check is a short answer.

Only the idea's own ways can give a hands-on check. An idea with no way that a walk can show gets a short answer in a text way. A walk passes `hands=False` when it has its share of hands-on checks (see `hands_most` in `journey`).

In [ ]:
#| export
CHECK_KIND = dict(demo='demo', order='order', code='code')
DEPTH = dict(short=2, demo=2, order=2, code=4, notebook=4)   # a notebook check is the code way with a library: it runs in Leela

def check_for(c, st, rng=None, explore:str='sometimes', prior=(), last:str='', hands:bool=True) -> AttrDict:
    "The way, kind, depth and reason for one check of concept `c`. A hands-on way of the idea comes before its text ways; `hands=False` leaves the hands-on ways out."
    own = [w for w in c.get('ways') or () if w in WALK_WAYS]
    ways = [w for w in own if w != last] or own
    hand = [w for w in ways if w in CHECK_KIND] if hands else []
    text = [w for w in ways if w not in CHECK_KIND] or [w for w in own if w not in CHECK_KIND] or [w for w in TEXT_WAYS if w not in CHECK_KIND]
    way, note = choose(st, c.get('subject', ''), False, rng, explore, prior, allowed=hand or text)
    kind = CHECK_KIND.get(way, 'short')
    depth = (2 if (c.get('floor') or 0) < 2 else 3) if kind == 'short' else DEPTH[kind]
    return AttrDict(way=way, kind=kind, depth=depth, why=note or f"{LABELS[way].lower()} {'fit' if way == 'words' else 'fits'} this idea")

A check takes a way of its idea. An idea with no way that a walk can show gets a text way.

In [ ]:
c = AttrDict(ways=['code', 'maths'], floor=0)
k = check_for(c, EMPTY, rng=random.Random(1))
assert (k.way, k.kind, k.depth) in {('code', 'code', 4), ('maths', 'short', 2)} and k.why
assert check_for(AttrDict(ways=['demo'], floor=0), EMPTY).kind == 'demo'
assert check_for(AttrDict(ways=['timeline'], floor=3), EMPTY).way in TEXT_WAYS   # no timeline view yet
assert check_for(AttrDict(ways=['words'], floor=3), EMPTY).depth == 3
assert all(check_for(AttrDict(ways=['demo', 'code'], floor=0), EMPTY, rng=rng, last='demo').way == 'code' for _ in range(20))

An idea that lists a hands-on way (demo, order or code) gets a check of that kind. This is true when a text way comes first in the list,
and when the profile likes a text way. An idea with only text ways still gets a short answer.

In [ ]:
assert all(check_for(AttrDict(ways=['words', 'example', 'order'], floor=0), EMPTY, random.Random(k), prior=['example']).kind == 'order' for k in range(50))
assert all(check_for(AttrDict(ways=['maths', 'demo'], floor=0), EMPTY, random.Random(k)).kind == 'demo' for k in range(50))
assert all(check_for(AttrDict(ways=['words', 'analogy'], floor=0), EMPTY, random.Random(k)).kind == 'short' for k in range(50))

An idea with no ways, or with no way that a walk can show, gets a short answer, never a code check. With `hands=False` (the walk has its share of hands-on checks), an idea gets a short answer in one of its text ways, else in any text way.

In [ ]:
assert all(check_for(AttrDict(ways=w, floor=0), EMPTY, random.Random(k)).kind == 'short' for w in ([], None, ['timeline']) for k in range(50))
assert all(check_for(AttrDict(ways=['code', 'maths'], floor=0), EMPTY, random.Random(k), hands=False).way == 'maths' for k in range(20))
assert all(check_for(AttrDict(ways=['demo'], floor=0), EMPTY, random.Random(k), hands=False).kind == 'short' for k in range(20))

## In plain words, and from the attempts

`methods_text` says what works, for a prompt and for the page. `method_stats` builds the counts again from the attempts, in order.

In [ ]:
#| export
def methods_text(st, subject:str='', n:int=3) -> str:
    "The methods with counts, best first, as one line; empty when nothing is counted."
    ms = {m: ab for m, ab in ((st.get('methods') or {}).get(subject or '*') or {}).items() if sum(ab) >= 1}
    if not ms: return ''
    top = sorted(ms, key=lambda m: -mean(posterior(st, m, subject)))[:n]
    bits = [f"{LABELS[m].lower()} {round(ms[m][0])} of {round(sum(ms[m]))}" + (' (better lately)' if improving(st, m) else '') for m in top]
    return 'Ways that work for you: ' + '; '.join(bits) + '.'

@patch
def method_stats(self:Learning, n:int=2000) -> dict:
    "Count every final attempt with a method again, oldest first, with the feedback on each attempt, and save the counts. Return the stats."
    st = {**EMPTY, 'calib': self.stats().get('calib') or {}}
    for a in reversed(self.attempts(limit=n)):
        if a.get('final') and a.get('method') and not a.get('dont_know'):
            c = self.concept(a.concept)
            st = credit(st, a.method, (c or {}).get('subject', ''), a.get('outcome', ''))
        for f in a.get('fb') or ():   # the verdicts on the demo or the lesson of this attempt
            if f.get('verdict') in FEEDBACK and f.get('way'): st = credit(st, f['way'], (self.concept(a.concept) or {}).get('subject', ''), *FEEDBACK[f['verdict']])
    with _LOCK: self.set_stats(**st)
    return st

The counts that `credit_method` keeps match a rebuild from the attempts.

In [ ]:
ml_ = temp_learning(); ml_.put('Ranking', 'Order items by a score.', 'ml')
for m, k in (('code', 'solid'), ('code', 'fragile'), ('maths', 'gap')):
    ml_.record('ranking', f'{m} {k}?', final=True, method=m, outcome=k); ml_.credit_method(m, 'ml', k); time.sleep(.01)
inc = ml_.stats()
test_eq(ml_.method_stats()['methods'], inc.methods)
assert methods_text(inc, 'ml').startswith('Ways that work for you: code 2 of 2')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()